In [ ]:
import torch #to create the tensors we will use to store the raw data
import torch.nn as nn #for module,linear,embedding classes
import torch.nn.functional as F #to access the softmax()
from torch.optim import Adam # to fit the neural network to data during backprop
from torch.utils.data import TensorDataset, DataLoader #trainingData
import lightning as L

In [6]:
token_to_id = {
    'what' :0,
    'is':1,
    'tranformers':2,
    'awesome':3,
    '<EOS>':4,
}
id_to_token = dict(map(reversed,token_to_id.items()))

In [ ]:
inputs = torch.tensor([
    [token_to_id['what']],
    [token_to_id['is']],
    [token_to_id['tranformers']],
    [token_to_id['<EOS>']],
    [token_to_id['awesome']],
    
    [token_to_id['tranformers']],
    [token_to_id['is']],
    [token_to_id['what']],
    [token_to_id['<EOS>']],
    [token_to_id['awesome']]
])

In [12]:
labels = torch.tensor([
    [token_to_id['is']],
    [token_to_id['tranformers']],
    [token_to_id['<EOS>']],
    [token_to_id['awesome']],
    [token_to_id['<EOS>']],
    
    [token_to_id['is']],
    [token_to_id['what']],
    [token_to_id['<EOS>']],
    [token_to_id['awesome']],
    [token_to_id['<EOS>']]
])

In [13]:
dataset = TensorDataset(inputs,labels)
dataloader = DataLoader(dataset)

## PositionEncoding class

### `__init__(self, d_model=2, max_len=6)`
- `d_model = 2` → each position's encoding vector has 2 numbers (kept small for illustration)
- `max_len = 6` → precompute positional encodings for positions 0 through 5

### Step 1: Create an empty table
```python
pe = torch.zeros(max_len, d_model)
```
Makes a 6×2 matrix of zeros — one row per position, one column per embedding dimension. This will be filled in with sin/cos values.

### Step 2: List out the positions
```python
position = torch.arange(0, max_len, 1).float().unsqueeze(1)
```
Creates `[0, 1, 2, 3, 4, 5]` as a column vector (shape 6×1), so it can be broadcast against the dimension indices later.

### Step 3: List out the dimension indices
```python
embedding_index = torch.arange(0, d_model, 2).float()
```
For `d_model=2`, this gives `[0]` — just the even dimension indices (0, 2, 4, ... in general). Each even index plus its odd neighbor forms one sin/cos pair.

### Step 4: Compute the denominator (the "frequency" for each pair)
```python
div_term = 1 / torch.tensor(10000.0) ** (embedding_index / d_model)
```
This is `1 / 10000^(2i/d_model)` — same formula discussed earlier, just written as a division instead of `pos / 10000^(...)`. It controls how fast each dimension pair oscillates: low dimensions oscillate fast, high dimensions oscillate slowly.

### Step 5: Fill in sin on even columns, cos on odd columns
```python
pe[:, 0::2] = torch.sin(position * div_term)
pe[:, 1::2] = torch.cos(position * div_term)
```
- `position * div_term` broadcasts the 6 positions against the frequency terms, giving a matrix of angle values
- `pe[:, 0::2]` selects all even-indexed columns (0, 2, 4...) and fills them with sin of those angles
- `pe[:, 1::2]` selects all odd-indexed columns (1, 3, 5...) and fills them with cos of those angles

### Step 6: Save it as a buffer
```python
self.register_buffer('pe', pe)
```
Stores `pe` as part of the model's state, but marks it as **not trainable** (no gradient updates) — it's a fixed lookup table, not a learned parameter. It will also move automatically to GPU/CPU along with the rest of the model.

### `forward(self, word_embeddings)`
```python
return word_embeddings + self.pe[:word_embeddings.size(0), :]
```
Takes the actual word embeddings as input, slices out as many rows of `pe` as there are tokens in the sequence, and adds them elementwise. This is the step that injects position information into each token's vector.

In [21]:
class PositionEncoding(nn.Module):
    def __init__(self,d_model=2,max_len=6):
        super().__init__()
        
        pe = torch.zeros(max_len,d_model) # combining them to a matrix 2x6 with full of zeros
        
        position = torch.arange(start=0,end = max_len,step=1).float().unsqueeze(1) 
        embedding_index = torch.arange(start=0,end=d_model,step=2).float()

        div_term = 1/torch.tensor(10000.0)**(embedding_index/d_model)

        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)

        self.register_buffer('pe',pe)

    def forward(self, word_embeddings):
        return word_embeddings + self.pe[:word_embeddings.size(0),:]

## Attention class

### `__init__(self, d_model=2)`
- `d_model = 2` → each token is represented by a vector of 2 numbers
- Creates three separate linear layers, one each for Query, Key, and Value:
```python
  self.W_q = nn.Linear(d_model, d_model, bias=False)
  self.W_k = nn.Linear(d_model, d_model, bias=False)
  self.W_v = nn.Linear(d_model, d_model, bias=False)
```
  Each one is just a learnable weight matrix (no bias term) that projects the input embeddings into Q, K, and V spaces.
- `self.row_dim = 0`, `self.col_dim = 1` — these just record which tensor dimension is "rows" (tokens) and which is "columns" (features), so later operations like transpose and softmax know which axis to act on.

---

### `forward(self, encodings_for_q, encodings_for_k, encodings_for_v, mask=None)`

This takes in three separate sets of embeddings (they can be the same for self-attention, or different for cross-attention) and runs the attention computation.

#### Step 1: Project into Q, K, V
```python
q = self.W_q(encodings_for_q)
k = self.W_k(encodings_for_k)
v = self.W_v(encodings_for_v)
```
Passes each input through its respective linear layer. Now you have separate Query, Key, and Value matrices.

#### Step 2: Compute similarity scores
```python
sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))
```
- `k.transpose(...)` flips K so its rows and columns are swapped (turns it into Kᵀ)
- `torch.matmul(q, k.T)` computes the dot product between every query and every key — this measures how "relevant" each key is to each query
- Result: a matrix of raw similarity scores, one for every (query, key) pair

#### Step 3: Scale the scores
```python
scaled_sims = sims / torch.tensor(k.size(self.col_dim)**0.5)
```
Divides by `√d_k` (the square root of the key dimension) to prevent the scores from growing too large, which would otherwise destabilize the softmax in the next step.

#### Step 4: Apply mask (optional)
```python
if mask is not None:
    scaled_sims = scaled_sims.masked_fill(mask=mask, value=-1e9)
```
If a mask is provided (e.g. a causal mask for decoder self-attention), this fills masked-out positions with a very large negative number. After softmax, those positions become essentially 0 — meaning the model can't "see" or attend to them. This is how causal masking prevents a token from attending to future tokens.


#### Step 5: Convert scores to attention weights
```python
attention_percents = F.softmax(scaled_sims, dim=self.col_dim)
```
Applies softmax along the column dimension, turning the raw scores into probabilities that sum to 1 across each row. This tells you, for each query, what percentage of "attention" it pays to each key.

#### Step 6: Compute weighted sum of values
```python
attention_scores = torch.matmul(attention_percents, v)
```
Multiplies the attention weights by the Value vectors. Each output row is now a weighted average of all the Value vectors, weighted by how much attention that query paid to each corresponding key.

#### Return
```python
return attention_scores
```
Returns the final context vectors — the actual output of the attention layer, one vector per query/token.

In [22]:
class Attention(nn.Module):
    def __init__(self,d_model=2):
        super().__init__()

        self.d_model = d_model

        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

        self.row_dim = 0
        self.col_dim = 1

    def forward(self,encodings_for_q,encodings_for_k,encodings_for_v,mask=None):
        q = self.W_q(encodings_for_q)
        k = self.W_k(encodings_for_k)
        v = self.W_v(encodings_for_v)

        sims = torch.matmul(q,k.transpose(dim0=self.row_dim,dim1=self.col_dim))
        scaled_sims = sims/torch.tensor(k.size(self.col_dim)**0.5)

        if mask is not None:
            scaled_sims = scaled_sims.masked_fill(mask=mask,value =-1e9)

        attention_percents = F.softmax(scaled_sims,dim=self.col_dim)

        attention_scores = torch.matmul(attention_percents,v)

        return attention_scores

## DecoderOnlyTransformer class

This wraps everything together into a complete (tiny) decoder-only transformer — the same architecture family as GPT — using PyTorch Lightning to handle training boilerplate.

### `__init__(self, num_token=4, d_model=2, max_len=6)`
- `num_token = 4` → vocabulary size (4 possible tokens)
- `d_model = 2` → each token embedding has 2 numbers
- `max_len = 6` → maximum sequence length supported

`L.seed_everything(seed=42)` fixes the random seed so weight initialization is reproducible — same results every run.

`self.we = nn.Embedding(...)` is the **word embedding** lookup table. Converts each token ID into a learnable `d_model`-sized vector.

`self.pe = PositionEncoding(...)` is the positional encoding module from earlier — adds fixed sin/cos position vectors to the word embeddings.

`self.self_attention = Attention(...)` is the self-attention block from earlier — computes Q, K, V and produces context vectors.

`self.fc_layer = nn.Linear(...)` is a final linear layer that projects the `d_model`-sized vectors back up to `num_token` size — these become the logits used to predict the next token over the vocabulary.

`self.loss = nn.CrossEntropyLoss()` is the standard loss function for classification — compares predicted token logits against the true next-token labels.

---

### `forward(self, token_ids)`

**Step 1 — Embed the tokens:** `word_embeddings = self.we(token_ids)` looks up each token ID's embedding vector.

**Step 2 — Add positional encoding:** `positional_encoded = self.pe(word_embeddings)` injects position information.

**Step 3 — Build the causal mask:**
```python
mask = torch.tril(torch.ones((token_ids.size(dim=0), token_ids.size(dim=0)), device=self.device))
mask = mask == 0
```
`torch.ones((T, T))` makes a T×T matrix of all 1s. `torch.tril(...)` keeps only the lower triangle (including diagonal), zeroing out everything above it:

for T=3:

[[1, 0, 0],

[1, 1, 0],

[1, 1, 1]]

Row `i` has 1s only up to column `i` — meaning token `i` can see itself and everything before it, but nothing after. `mask == 0` flips this into a boolean mask where `True` marks the positions that should be blocked (the upper triangle, i.e. future tokens):


[[False, True,  True ],

[False, False, True ],

[False, False, False]]


This is fed into the `Attention` class, which fills these `True` positions with `-1e9` before softmax — enforcing that each token can only attend to itself and earlier tokens, never future ones. This is what makes it a **decoder-only** (autoregressive) model.

**Step 4 — Run self-attention:**
```python
self_attention_values = self.self_attention(positional_encoded,
                                              positional_encoded,
                                              positional_encoded,
                                              mask=mask)
```
Same tensor is passed as Q, K, and V inputs — this is **self**-attention, the model attending to its own sequence. The causal mask ensures no peeking at future tokens.

**Step 5 — Residual connection:** `residual_connection_values = positional_encoded + self_attention_values` adds the attention output back to the original positional-encoded input — the skip connection that helps gradients flow and preserves the original signal alongside the attention-derived context. (Note: a real transformer would also apply LayerNorm here; this simplified version skips it.)

**Step 6 — Project to vocabulary logits:** `fc_layer_output = self.fc_layer(residual_connection_values)` maps each token's `d_model`-sized vector to `num_token` logits — a raw, unnormalized score for every word in the vocabulary, predicting what the next token should be.

---

### `configure_optimizers(self)`
`return Adam(self.parameters(), lr=0.1)` tells Lightning to use the Adam optimizer with a learning rate of `0.1` to update all the model's trainable parameters (embedding table, Q/K/V weight matrices, final linear layer) during training.

---

### `training_step(self, batch, batch_idx)`
```python
input_tokens, labels = batch
output = self.forward(input_tokens[0])
loss = self.loss(output, labels[0])
return loss
```
Lightning calls this automatically once per training batch: unpacks the batch into input tokens and target labels, runs the forward pass on `input_tokens[0]` (first item in the batch), computes cross-entropy loss between the predicted logits and the true labels, and returns the loss, which Lightning uses to run backpropagation and update weights automatically.

In [36]:
class DecoderOnlyTransformer(L.LightningModule):
    def __init__(self,num_token=4,d_model=2,max_len=6):
        super().__init__()

        L.seed_everything(seed=42)

        self.we = nn.Embedding(num_embeddings=num_token,embedding_dim=d_model)
        self.pe = PositionEncoding(d_model=d_model,max_len=max_len)
        self.self_attention = Attention(d_model=d_model)

        self.fc_layer = nn.Linear(in_features=d_model,out_features=num_token)
        self.loss = nn.CrossEntropyLoss()

    def forward(self,token_ids):
        word_embeddings = self.we(token_ids)
        positional_encoded = self.pe(word_embeddings)

        mask = torch.tril(torch.ones((token_ids.size(dim=0),token_ids.size(dim=0)),device=self.device))
        mask = mask == 0

        self_attention_values = self.self_attention(positional_encoded,
                                                    positional_encoded,
                                                    positional_encoded,
                                                    mask=mask)

        residual_connection_values = positional_encoded + self_attention_values

        fc_layer_output = self.fc_layer(residual_connection_values)

        return fc_layer_output

    def configure_optimizers(self):
        return Adam(self.parameters(),lr=0.1)

    def training_step(self,batch,batch_idx):
        input_tokens , labels = batch
        output = self.forward(input_tokens[0])
        loss = self.loss(output,labels[0])

        return loss


## Inference / generation code

This block runs the trained (or untrained, in this case) model to generate a sequence of tokens, one at a time — the standard autoregressive generation loop.

### Step 1: Create the model
```python
model = DecoderOnlyTransformer(num_token=len(token_to_id), d_model=2, max_len=6)
```
Instantiates the transformer. `num_token` is set dynamically based on however many unique tokens exist in the `token_to_id` vocabulary dictionary.

### Step 2: Build the initial input
```python
model_input = torch.tensor(
    [token_to_id["what"], 
    token_to_id["is"], 
    token_to_id["tranformers"], 
    token_to_id["<EOS>"]]
)
```
Converts the prompt "what is tranformers \<EOS\>" into a tensor of token IDs by looking each word up in the vocabulary. `<EOS>` here marks the end of the **input** prompt — the model will start generating its own tokens after this.

```python
input_length = model_input.size(dim=0)
```
Stores how many tokens are in the prompt (4 in this case) — used later to know where generation should start.

### Step 3: Get the first prediction
```python
predictions = model(model_input)
predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
```
- Runs the full prompt through the model in one forward pass
- `predictions` has shape `(seq_len, num_token)` — one row of logits per input position
- `predictions[-1, :]` takes only the **last row** — the logits predicted after seeing the entire prompt, i.e. the model's guess for what comes next
- `torch.argmax(...)` picks the token ID with the highest score (greedy decoding — always picks the single most likely next token, no sampling)

```python
predicted_ids = predicted_id
```
Starts a running list of all generated token IDs, beginning with this first prediction.

### Step 4: Generation loop
```python
max_length = 6
for i in range(input_length, max_length):
    if (predicted_id == token_to_id["<EOS>"]):
        break
    model_input = torch.cat((model_input, predicted_id))
    predictions = model(model_input)
    predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
    predicted_ids = torch.cat((predicted_ids, predicted_id))
```
Loops from where the prompt ended (`input_length`) up to `max_length` total tokens, generating one new token per iteration:

1. **Stop check:** if the most recently predicted token is `<EOS>`, stop generating — the model has signaled it's done
2. **Extend input:** append the newly predicted token onto `model_input`, so the next forward pass sees the prompt **plus** everything generated so far
3. **Re-run the model:** feed the extended sequence back through the model from scratch
4. **Predict again:** take the logits from the last position and argmax to get the next token
5. **Record it:** append this new token ID onto `predicted_ids`, the running output list

This is the classic (inefficient but simple) autoregressive generation pattern — each new token depends on everything generated before it, and the entire sequence is reprocessed every step. Production-grade systems use a **KV cache** to avoid recomputing earlier tokens' attention, but this version recomputes everything for clarity.

### Step 5: Print the result
```python
print("Predicted Tokens:\n") 
for id in predicted_ids: 
    print("\t", id_to_token[id.item()])
```
Converts each generated token ID back into its word using the reverse lookup dictionary `id_to_token`, and prints them out one per line.

---

**Note:** since this model hasn't been trained (no `trainer.fit(...)` call shown), the weights are just random initializations — so the generated tokens will be essentially meaningless. This code demonstrates the *mechanics* of generation, not a coherent output.

In [37]:
model = DecoderOnlyTransformer(num_token=len(token_to_id),d_model=2,max_len=6)
model_input = torch.tensor(
    [token_to_id["what"], 
    token_to_id["is"], 
    token_to_id["tranformers"], 
    token_to_id["<EOS>"]]
)
input_length = model_input.size(dim=0)

predictions = model(model_input)
predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])

predicted_ids = predicted_id

max_length = 6
for i in range(input_length,max_length):
    if (predicted_id == token_to_id["<EOS>"]):
        break
    model_input = torch.cat((model_input,predicted_id))
    predictions = model(model_input)
    predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
    predicted_ids = torch.cat((predicted_ids, predicted_id))
        
## Now printout the predicted output phrase.
print("Predicted Tokens:\n") 
for id in predicted_ids: 
    print("\t", id_to_token[id.item()])


Seed set to 42


Predicted Tokens:

	 <EOS>


## Training the model

```python
trainer = L.Trainer(max_epochs=30)
trainer.fit(model, train_dataloaders=dataloader)
```

### `L.Trainer(max_epochs=30)`
Creates a PyTorch Lightning `Trainer` object — this handles all the training boilerplate automatically (the training loop, moving data/model to the right device, calling `backward()`, stepping the optimizer, logging, etc.) so you don't have to write it manually.

`max_epochs=30` tells it to run through the entire training dataset 30 times.

### `trainer.fit(model, train_dataloaders=dataloader)`
Starts training. Under the hood, for each epoch, for each batch from `dataloader`, Lightning will:

1. Call `model.training_step(batch, batch_idx)` — this is the method defined earlier, which runs the forward pass and computes the loss
2. Call `loss.backward()` — computes gradients for every trainable parameter (the embedding table, the Q/K/V weight matrices in `Attention`, and the final `fc_layer`) via backpropagation
3. Call `optimizer.step()` — uses the Adam optimizer (defined in `configure_optimizers`) to update all the weights based on those gradients
4. Call `optimizer.zero_grad()` — resets gradients before the next batch

After this call finishes, `model`'s weights are no longer random — they've been updated to (hopefully) better predict the next token given the training examples in `dataloader`. Re-running the earlier generation/inference code afterward should now produce more meaningful output instead of random tokens.

In [29]:
trainer = L.Trainer(max_epochs=30)
trainer.fit(model, train_dataloaders=dataloader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name           | Type             | Params | Mode  | FLOPs
--------------------------------------------------------------------
0 | we             | Embedding        | 10     | train | 0    
1 | pe             | PositionEncoding | 0      | train | 0    
2 | self_attention | Attention        | 12     | train | 0    
3 | fc_layer       | Linear           | 15     | train | 0    
4 | loss           | CrossEntropyLoss | 0      | train | 0    
--------------------------------------

Epoch 29: 100%|██████████| 10/10 [00:00<00:00, 243.02it/s, v_num=0]

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 10/10 [00:00<00:00, 209.93it/s, v_num=0]


# Inference / generate code again with trained model

In [33]:
model_input = torch.tensor([token_to_id["what"], 
                            token_to_id["is"], 
                            token_to_id["tranformers"], 
                            token_to_id["<EOS>"]])
input_length = model_input.size(dim=0)

predictions = model(model_input) 
predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
predicted_ids = predicted_id

for i in range(input_length, max_length):
    if (predicted_id == token_to_id["<EOS>"]): # if the prediction is <EOS>, then we are done
        break
    
    model_input = torch.cat((model_input, predicted_id))
    
    predictions = model(model_input) 
    predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
    predicted_ids = torch.cat((predicted_ids, predicted_id))
        
print("Predicted Tokens:\n") 
for id in predicted_ids: 
    print("\t", id_to_token[id.item()])

Predicted Tokens:

	 awesome
	 is
	 <EOS>


In [ ]:
model_input = torch.tensor([token_to_id["tranformers"], 
                            token_to_id["is"], 
                            token_to_id["what"], 
                            token_to_id["<EOS>"]])
input_length = model_input.size(dim=0)

predictions = model(model_input) 
predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
predicted_ids = predicted_id

for i in range(input_length, max_length):
    if (predicted_id == token_to_id["<EOS>"]):
        break
    
    model_input = torch.cat((model_input, predicted_id))
    
    predictions = model(model_input) 
    predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
    predicted_ids = torch.cat((predicted_ids, predicted_id))
        
print("Predicted Tokens:\n") 
for id in predicted_ids: 
    print("\t", id_to_token[id.item()])

Predicted Tokens:

	 awesome
	 is
	 <EOS>
